# AI-316 Lab 03 - Computer Vision Prototyping
Run `pip install -r requirements.txt` first. Put `utils.py` in the same folder.

## Task 1: Traffic preprocessing

In [ ]:
%matplotlib inline
"""Task 1 - Automated preprocessing pipeline for traffic camera footage.

Steps: imread -> letterbox resize to 640x640 (aspect ratio kept) ->
normalise [0,255] -> [0.0,1.0] -> side-by-side matplotlib display + tensor stats.
"""
import argparse
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np

from utils import OUT_DIR, load_image

TARGET = 640


def letterbox(img, size=TARGET, color=(114, 114, 114)):
    """Resize keeping aspect ratio, then pad to a size x size square."""
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(img, (new_w, new_h), interpolation=interp)

    top = (size - new_h) // 2
    bottom = size - new_h - top
    left = (size - new_w) // 2
    right = size - new_w - left
    padded = cv2.copyMakeBorder(resized, top, bottom, left, right,
                                cv2.BORDER_CONSTANT, value=color)
    return padded, scale, (left, top)


def normalize(img_uint8):
    """uint8 [0,255] -> float32 [0.0,1.0]."""
    return img_uint8.astype(np.float32) / 255.0


def describe(name, arr):
    print(f"{name:<22} shape={arr.shape}  dtype={arr.dtype}  "
          f"min={arr.min():.4f}  max={arr.max():.4f}  mean={arr.mean():.4f}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", default="traffic.jpg",
                        help="high-resolution traffic scene image")
    args, _ = parser.parse_known_args()

    # 1. Load
    bgr = load_image(args.image, fallback="bus.jpg")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # 2. Letterbox resize 640x640
    letter, scale, (padx, pady) = letterbox(rgb)

    # 3. Normalise
    norm = normalize(letter)
    tensor = np.transpose(norm, (2, 0, 1))[None]  # (1,3,640,640) NCHW model input

    print("=== Pixel tensor statistics ===")
    describe("Original (uint8)", rgb)
    describe("Letterboxed (uint8)", letter)
    describe("Normalised (float32)", norm)
    describe("Model tensor NCHW", tensor)
    print(f"scale={scale:.4f}, padding (left,top)=({padx},{pady})")

    # 4. Display
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
    axes[0].imshow(rgb)
    axes[0].set_title(f"Original ({rgb.shape[1]}x{rgb.shape[0]})")
    axes[1].imshow(letter)
    axes[1].set_title("Letterboxed 640x640 (uint8)")
    axes[2].imshow(norm)
    axes[2].set_title(f"Normalised float32 [{norm.min():.1f}, {norm.max():.1f}]")
    for ax in axes:
        ax.set_xlabel("Width (pixels)")
        ax.set_ylabel("Height (pixels)")
    axes[2].text(8, 625, f"shape={norm.shape}  min={norm.min():.2f}  max={norm.max():.2f}",
                 color="yellow", fontsize=9, backgroundcolor="black")
    fig.suptitle("Task 1: Traffic camera preprocessing pipeline", fontsize=14)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, "task1_result.png")
    plt.savefig(out, dpi=150)
    print("Saved:", out)
    plt.show()


if __name__ == "__main__":
    main()


## Task 2: Defect detection

In [ ]:
%matplotlib inline
"""Task 2 - Industrial defect (rust) detection via colour spaces + thresholding.

BGR -> HSV / LAB, cv2.inRange mask, cv2.threshold refinement, cv2.bitwise_and.
If no image is given, a synthetic metal plate with rust patches is generated.
"""
import argparse
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np

from utils import OUT_DIR

# HSV bounds for rust / orange-brown (OpenCV: H 0-179, S 0-255, V 0-255)
LOWER_HSV = np.array([5, 100, 50])
UPPER_HSV = np.array([25, 255, 255])


def make_synthetic_product(h=480, w=640, seed=0):
    rng = np.random.default_rng(seed)
    img = np.full((h, w, 3), (170, 170, 175), np.uint8)               # grey metal
    cv2.rectangle(img, (50, 50), (w - 50, h - 50), (150, 150, 155), -1)
    for cx, cy in [(90, 90), (w - 90, 90), (90, h - 90), (w - 90, h - 90)]:
        cv2.circle(img, (cx, cy), 18, (110, 110, 115), -1)             # bolts
    for cx, cy, a, b in [(220, 200, 55, 30), (430, 300, 70, 40), (300, 380, 35, 25)]:
        cv2.ellipse(img, (cx, cy), (a, b), int(rng.integers(0, 180)), 0, 360,
                    (30, 85, 165), -1)                                  # rust (BGR)
    img = cv2.GaussianBlur(img, (7, 7), 0)
    noise = rng.normal(0, 5, img.shape)
    return np.clip(img + noise, 0, 255).astype(np.uint8)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", default=None, help="industrial product image")
    args, _ = parser.parse_known_args()

    if args.image and os.path.exists(args.image):
        bgr = cv2.imread(args.image)
    else:
        print("[i] No image supplied -> using synthetic rusted metal plate")
        bgr = make_synthetic_product()

    # 1. Colour-space conversion
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)

    # 2-3. Range mask -> smooth -> binary threshold -> morphology
    mask = cv2.inRange(hsv, LOWER_HSV, UPPER_HSV)
    blurred = cv2.GaussianBlur(mask, (5, 5), 0)
    _, refined = cv2.threshold(blurred, 127, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    refined = cv2.morphologyEx(refined, cv2.MORPH_OPEN, kernel)
    refined = cv2.morphologyEx(refined, cv2.MORPH_CLOSE, kernel)

    # 4. Isolate defect
    isolated = cv2.bitwise_and(bgr, bgr, mask=refined)

    area = 100.0 * np.count_nonzero(refined) / refined.size
    contours, _ = cv2.findContours(refined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"Defect area: {area:.2f}% of image | regions found: {len(contours)}")

    # Plot grid
    H, S, V = cv2.split(hsv)
    L, A, B = cv2.split(lab)
    panels = [
        ("Original (BGR->RGB)", cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), None),
        ("HSV - Hue", H, "hsv"), ("HSV - Saturation", S, "gray"), ("HSV - Value", V, "gray"),
        ("LAB - L", L, "gray"), ("LAB - a", A, "gray"), ("LAB - b", B, "gray"),
        ("inRange mask", mask, "gray"),
        ("Refined mask (threshold+morph)", refined, "gray"),
        ("Isolated defect (bitwise_and)", cv2.cvtColor(isolated, cv2.COLOR_BGR2RGB), None),
    ]
    fig, axes = plt.subplots(3, 4, figsize=(16, 11))
    for ax, (title, img, cmap) in zip(axes.ravel(), panels):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.set_xlabel("x (px)")
        ax.set_ylabel("y (px)")
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")
    fig.suptitle(f"Task 2: Rust detection  |  HSV range {LOWER_HSV.tolist()} - "
                 f"{UPPER_HSV.tolist()}  |  defect area {area:.2f}%", fontsize=13)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, "task2_result.png")
    plt.savefig(out, dpi=130)
    print("Saved:", out)
    plt.show()


if __name__ == "__main__":
    main()


## Task 3: Medical edges

In [ ]:
%matplotlib inline
"""Task 3 - Noise reduction + edge detection for medical imaging.

Grayscale load -> (add synthetic noise) -> Gaussian vs Median filtering ->
Sobel X/Y + gradient magnitude -> Canny with tuned hysteresis thresholds ->
quantitative comparison table (PSNR, edge precision/recall/F1 vs clean image).
"""
import argparse
import csv
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np

from utils import OUT_DIR


def make_phantom(size=512):
    """Synthetic chest-like phantom, used when no scan is supplied."""
    img = np.zeros((size, size), np.uint8)
    cv2.ellipse(img, (256, 256), (210, 240), 0, 0, 360, 110, -1)       # body
    cv2.ellipse(img, (180, 250), (70, 130), 0, 0, 360, 40, -1)          # lung L
    cv2.ellipse(img, (332, 250), (70, 130), 0, 0, 360, 40, -1)          # lung R
    cv2.circle(img, (256, 290), 28, 200, -1)                            # spine
    cv2.circle(img, (256, 330), 40, 160, -1)                            # heart-ish
    return cv2.GaussianBlur(img, (3, 3), 0)


def add_noise(img, sigma=18, sp=0.03, seed=0):
    rng = np.random.default_rng(seed)
    noisy = np.clip(img + rng.normal(0, sigma, img.shape), 0, 255).astype(np.uint8)
    r = rng.random(img.shape)
    noisy[r < sp / 2] = 0
    noisy[r > 1 - sp / 2] = 255
    return noisy


def sobel_edges(gray):
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    mag8 = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return gx, gy, mag, mag8


def edge_scores(edges, ref, tol=2):
    """Precision / recall / F1 of an edge map vs reference, with pixel tolerance."""
    k = np.ones((2 * tol + 1, 2 * tol + 1), np.uint8)
    ref_d, edg_d = cv2.dilate(ref, k), cv2.dilate(edges, k)
    n_e, n_r = np.count_nonzero(edges), np.count_nonzero(ref)
    p = np.count_nonzero((edges > 0) & (ref_d > 0)) / max(n_e, 1)
    r = np.count_nonzero((ref > 0) & (edg_d > 0)) / max(n_r, 1)
    f1 = 2 * p * r / max(p + r, 1e-9)
    return p, r, f1


def tune_canny(gray, ref, grid=((20, 60), (30, 90), (50, 150), (70, 200), (100, 200))):
    best = None
    for lo, hi in grid:
        f1 = edge_scores(cv2.Canny(gray, lo, hi), ref)[2]
        if best is None or f1 > best[2]:
            best = (lo, hi, f1)
    return best[:2]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", default=None, help="X-ray / CT image")
    parser.add_argument("--no-noise", action="store_true",
                        help="do not add synthetic noise (image already noisy)")
    parser.add_argument("--sigma", type=float, default=18)
    args, _ = parser.parse_known_args()

    # 1. Load grayscale
    if args.image and os.path.exists(args.image):
        clean = cv2.imread(args.image, cv2.IMREAD_GRAYSCALE)
    else:
        print("[i] No scan supplied -> using synthetic phantom")
        clean = make_phantom()
    noisy = clean if args.no_noise else add_noise(clean, sigma=args.sigma)

    # 2. Filters
    variants = {
        "No filter": noisy,
        "Gaussian 5x5": cv2.GaussianBlur(noisy, (5, 5), 0),
        "Median 5": cv2.medianBlur(noisy, 5),
    }

    # Reference edges from the clean image (ground-truth proxy)
    ref_edges = cv2.Canny(clean, 50, 150)

    rows, plots = [], []
    for name, img in variants.items():
        gx, gy, mag, mag8 = sobel_edges(img)                     # 3. Sobel
        lo, hi = tune_canny(img, ref_edges)                      # 4. tuned Canny
        canny = cv2.Canny(img, lo, hi)
        p, r, f1 = edge_scores(canny, ref_edges)
        rows.append({
            "filter": name,
            "PSNR_dB": round(cv2.PSNR(clean, img), 2),
            "sharpness_LapVar": round(cv2.Laplacian(img, cv2.CV_64F).var(), 1),
            "sobel_mean_mag": round(float(mag.mean()), 2),
            "canny_T_low": lo, "canny_T_high": hi,
            "canny_edge_px": int(np.count_nonzero(canny)),
            "precision": round(p, 3), "recall": round(r, 3), "F1": round(f1, 3),
        })
        plots.append((name, img, mag8, canny))

    # Table
    keys = list(rows[0].keys())
    print("\n=== Quantitative comparison (edges vs clean-image reference) ===")
    print(" | ".join(f"{k:>16}" for k in keys))
    for r in rows:
        print(" | ".join(f"{str(r[k]):>16}" for k in keys))
    csv_path = os.path.join(OUT_DIR, "task3_comparison.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print("Saved:", csv_path)

    # Plot
    fig, axes = plt.subplots(len(plots) + 1, 3, figsize=(13, 16))
    axes[0, 0].imshow(clean, cmap="gray"); axes[0, 0].set_title("Clean / original")
    axes[0, 1].imshow(noisy, cmap="gray"); axes[0, 1].set_title("Noisy input")
    axes[0, 2].imshow(ref_edges, cmap="gray"); axes[0, 2].set_title("Reference Canny (clean)")
    for i, (name, img, mag8, canny) in enumerate(plots, start=1):
        axes[i, 0].imshow(img, cmap="gray"); axes[i, 0].set_title(f"{name}")
        axes[i, 1].imshow(mag8, cmap="gray"); axes[i, 1].set_title(f"Sobel magnitude - {name}")
        axes[i, 2].imshow(canny, cmap="gray")
        axes[i, 2].set_title(f"Canny ({rows[i-1]['canny_T_low']},{rows[i-1]['canny_T_high']}) "
                             f"F1={rows[i-1]['F1']}")
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Task 3: Noise reduction & edge detection", fontsize=14)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, "task3_result.png")
    plt.savefig(out, dpi=110)
    print("Saved:", out)
    plt.show()


if __name__ == "__main__":
    main()


## Task 4: YOLO benchmark

In [ ]:
%matplotlib inline
"""Task 4 - YOLOv8n vs YOLOv8m benchmarking (latency, class counts, confidence)."""
import argparse
import csv
import os
import time
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

from utils import OUT_DIR, load_image


def detect(model, img, conf=0.25, runs=5):
    """Warm-up once, then average latency over `runs`; return detections of last run."""
    model.predict(img, conf=conf, verbose=False)  # warm-up (not timed)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        res = model.predict(img, conf=conf, verbose=False)[0]
        times.append((time.perf_counter() - t0) * 1000)

    dets = []
    for b in res.boxes:
        x1, y1, x2, y2 = (int(v) for v in b.xyxy[0].tolist())
        cid = int(b.cls[0])
        dets.append({"box": (x1, y1, x2, y2), "conf": float(b.conf[0]),
                     "cls_id": cid, "label": res.names[cid]})
    return dets, float(np.mean(times))


def draw(img, dets, color, title):
    out = img.copy()
    for d in dets:
        x1, y1, x2, y2 = d["box"]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        txt = f"{d['label']} {d['conf']:.2f}"
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(out, (x1, max(0, y1 - th - 6)), (x1 + tw + 4, y1), color, -1)
        cv2.putText(out, txt, (x1 + 2, max(th, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.rectangle(out, (0, 0), (out.shape[1], 30), (0, 0, 0), -1)
    cv2.putText(out, title, (8, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    return out


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", default=None, help="test image from your dataset")
    parser.add_argument("--models", nargs=2, default=["yolov8n.pt", "yolov8m.pt"])
    parser.add_argument("--conf", type=float, default=0.25)
    parser.add_argument("--runs", type=int, default=5)
    args, _ = parser.parse_known_args()

    img = load_image(args.image, fallback="bus.jpg")
    colors = [(0, 200, 0), (0, 0, 230)]
    results, rendered = [], []

    for name, color in zip(args.models, colors):
        model = YOLO(name)                               # 1. load
        dets, ms = detect(model, img, args.conf, args.runs)   # 2-3. infer + extract
        counts = Counter(d["label"] for d in dets)
        avg_conf = float(np.mean([d["conf"] for d in dets])) if dets else 0.0
        results.append({"model": name, "latency_ms": round(ms, 1),
                        "detections": len(dets), "unique_classes": len(counts),
                        "avg_confidence": round(avg_conf, 3),
                        "class_counts": dict(counts)})
        print(f"\n--- {name} ---")
        for d in dets:
            print(f"  {d['label']:<12} conf={d['conf']:.2f} box={d['box']}")
        rendered.append(draw(img, dets, color, f"{name} | {ms:.1f} ms | {len(dets)} objs"))  # 4.

    # Comparison table
    print("\n=== Comparison table ===")
    hdr = f"{'Model':<12}{'Latency (ms)':>14}{'Detections':>12}{'Classes':>9}{'Avg conf':>10}  Class counts"
    print(hdr)
    print("-" * len(hdr))
    for r in results:
        print(f"{r['model']:<12}{r['latency_ms']:>14}{r['detections']:>12}"
              f"{r['unique_classes']:>9}{r['avg_confidence']:>10}  {r['class_counts']}")

    csv_path = os.path.join(OUT_DIR, "task4_comparison.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=results[0].keys())
        w.writeheader()
        w.writerows(results)

    # Dual visual output
    fig, axes = plt.subplots(1, 2, figsize=(15, 8))
    for ax, im, r in zip(axes, rendered, results):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{r['model']}  ({r['latency_ms']} ms, avg conf {r['avg_confidence']})")
        ax.axis("off")
    plt.tight_layout()
    out = os.path.join(OUT_DIR, "task4_result.png")
    plt.savefig(out, dpi=120)
    print("Saved:", out, "and", csv_path)
    plt.show()


if __name__ == "__main__":
    main()


## Task 5: Retail inventory

In [ ]:
%matplotlib inline
"""Task 5 - Retail inventory: class-filtered YOLO detection + item counting.

Keeps only the target class IDs (COCO: 39 = bottle, 47 = apple), counts items
with confidence >= 0.50 and prints an on-image summary banner.
"""
import argparse
import os

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

from utils import OUT_DIR, load_image


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", default="shelf.jpg", help="crowded retail shelf image")
    parser.add_argument("--model", default="yolov8m.pt")
    parser.add_argument("--classes", type=int, nargs="+", default=[39],
                        help="COCO target class IDs (39=bottle, 47=apple)")
    parser.add_argument("--conf", type=float, default=0.50)
    args, _ = parser.parse_known_args()

    img = load_image(args.image, fallback="bus.jpg")
    model = YOLO(args.model)

    # 1. Run YOLO on everything with a low threshold, filter manually afterwards
    res = model.predict(img, conf=0.10, verbose=False)[0]

    # 2-3. Programmatic class + confidence filtering
    targets = []
    for b in res.boxes:
        cid, conf = int(b.cls[0]), float(b.conf[0])
        if cid in args.classes and conf >= args.conf:
            x1, y1, x2, y2 = (int(v) for v in b.xyxy[0].tolist())
            targets.append((x1, y1, x2, y2, conf, res.names[cid]))

    names = sorted({res.names[c] for c in args.classes})
    label = "/".join(names)
    print(f"Detected {len(res.boxes)} objects total, kept {len(targets)} "
          f"target item(s) [{label}] with conf >= {args.conf}")

    # 4. Overlay boxes + banner
    out = img.copy()
    for x1, y1, x2, y2, conf, name in targets:
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 220, 0), 2)
        cv2.putText(out, f"{name} {conf:.2f}", (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 0), 1, cv2.LINE_AA)
    banner = f"Target Item Count: {len(targets)}"
    cv2.rectangle(out, (0, 0), (out.shape[1], 45), (0, 0, 0), -1)
    cv2.putText(out, banner, (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (0, 255, 255), 2, cv2.LINE_AA)

    path = os.path.join(OUT_DIR, "task5_result.png")
    cv2.imwrite(path, out)
    print("Saved:", path)
    plt.figure(figsize=(11, 8))
    plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    plt.title(f"Task 5: {banner}")
    plt.axis("off")
    plt.show()


if __name__ == "__main__":
    main()


## Task 6: ROI intrusion

In [ ]:
%matplotlib inline
"""Task 6 - Security intrusion alerting with a polygon ROI.

YOLO person detection -> bbox centroid -> cv2.pointPolygonTest against ROI.
ROI red + "INTRUSION DETECTED" if any person centroid is inside, else green.
"""
import argparse
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

from utils import OUT_DIR, load_image

# default ROI as fractions of (width, height) so it fits any image size
DEFAULT_ROI = [(0.15, 0.35), (0.85, 0.35), (0.95, 0.95), (0.05, 0.95)]


def parse_roi(text, w, h):
    """--roi "100,200;400,200;400,500;100,500" in pixel coordinates."""
    if not text:
        return np.array([(int(x * w), int(y * h)) for x, y in DEFAULT_ROI], np.int32)
    return np.array([tuple(int(v) for v in p.split(",")) for p in text.split(";")], np.int32)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", default="surveillance.jpg")
    parser.add_argument("--model", default="yolov8n.pt")
    parser.add_argument("--roi", default=None, help='pixel polygon "x,y;x,y;x,y;..."')
    parser.add_argument("--conf", type=float, default=0.40)
    args, _ = parser.parse_known_args()

    img = load_image(args.image, fallback="bus.jpg")
    h, w = img.shape[:2]
    roi = parse_roi(args.roi, w, h)                                     # 1. ROI polygon

    res = YOLO(args.model).predict(img, conf=args.conf, classes=[0], verbose=False)[0]

    intrusion = False
    out = img.copy()
    for b in res.boxes:                                                  # 2. persons
        x1, y1, x2, y2 = (int(v) for v in b.xyxy[0].tolist())
        xc, yc = (x1 + x2) // 2, (y1 + y2) // 2                          # centroid
        inside = cv2.pointPolygonTest(roi.reshape(-1, 1, 2), (float(xc), float(yc)), False) >= 0  # 3.
        intrusion |= inside
        color = (0, 0, 255) if inside else (255, 200, 0)
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.circle(out, (xc, yc), 5, color, -1)
        cv2.putText(out, f"person {float(b.conf[0]):.2f}", (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    # 4. ROI colour + alert text
    roi_color = (0, 0, 255) if intrusion else (0, 255, 0)                # BGR red / green
    cv2.polylines(out, [roi.reshape(-1, 1, 2)], True, roi_color, 3)
    status = "INTRUSION DETECTED" if intrusion else "Area Clear"
    cv2.rectangle(out, (0, 0), (w, 50), (0, 0, 0), -1)
    cv2.putText(out, status, (10, 36), cv2.FONT_HERSHEY_SIMPLEX, 1.1, roi_color, 3, cv2.LINE_AA)

    print(f"Persons detected: {len(res.boxes)} | status: {status}")
    path = os.path.join(OUT_DIR, "task6_result.png")
    cv2.imwrite(path, out)
    print("Saved:", path)
    plt.figure(figsize=(9, 10))
    plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    plt.title(f"Task 6: {status}")
    plt.axis("off")
    plt.show()


if __name__ == "__main__":
    main()
